# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Agha314/FLyRank-Task-1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

Random Forest chosen over a linear model because content-decline signals likely involve non-linear interactions (e.g. word_count and position_std combined) that a single formula can't capture. Baseline (Week 4) scored 52% Precision@50 on the full dataset (base rate 8.75%) — but that mixed real-volume and insufficient-volume pages. This section restricts to real-volume pages (≥100 impressions) so baseline and RF can be compared on identical, fair ground.

In [1]:
# ---- Imports ----
import duckdb
from google.colab import userdata
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

# ---- Connection ----
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_Token')}')")

pdlink = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
cplink = 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'

# ---- Step 1: March performance, aggregated to page-level, joined with content properties ----
query = f"""
WITH monthly_perf AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS total_gsc_impressions_mar,
        SUM(gsc_clicks) AS total_gsc_clicks_mar,
        AVG(gsc_avg_position) AS avg_gsc_position_mar,
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions,
        STDDEV(gsc_avg_position) AS position_std_mar
    FROM read_parquet('{pdlink}')
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    mp.*,
    cp.* EXCLUDE (client_hash_id, content_hash_id)
FROM monthly_perf AS mp
JOIN read_parquet('{cplink}') AS cp
    ON mp.client_hash_id = cp.client_hash_id
    AND mp.content_hash_id = cp.content_hash_id
"""
merged = con.sql(query).df()
print("After join:", merged.shape)

# ---- Step 2: Label (is_declining) — for target/evaluation ONLY, never a feature ----
label_query = f"""
    SELECT
        client_hash_id,
        content_hash_id,
        CASE WHEN SUM(gsc_clicks) FILTER (WHERE report_date > DATE '2026-03-15')
                  < SUM(gsc_clicks) FILTER (WHERE report_date <= DATE '2026-03-15')
             THEN 1 ELSE 0 END AS is_declining
    FROM read_parquet('{pdlink}')
    GROUP BY client_hash_id, content_hash_id
"""
labels = con.sql(label_query).df()
merged = merged.merge(labels, on=['client_hash_id', 'content_hash_id'], how='left')

# ---- Step 3: Age features from dim_content dates ----
ref_date = pd.Timestamp('2026-03-31')
merged['content_age_days'] = (ref_date - merged['content_created_date']).dt.days
merged['days_since_last_update'] = (ref_date - merged['content_updated_date']).dt.days

# ---- Step 4: Drop columns decided against (product-flag risk, ~87% missing) ----
merged = merged.drop(columns=['last_optimized_date', 'optimization_eligible_date'], errors='ignore')

# ---- Step 5: Clean dataset ----
merged = merged[(merged['is_deleted'] == False) & (merged['is_published'] == True)]
merged = merged[merged['total_gsc_impressions_mar'] >= 100]   # real-volume only

print("Final dataset shape after cleaning:", merged.shape)
print("Label balance:\n", merged['is_declining'].value_counts(normalize=True))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

After join: (331437, 31)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Final dataset shape after cleaning: (101409, 32)
Label balance:
 is_declining
0    0.735763
1    0.264237
Name: proportion, dtype: float64


## 2. Split design

Split by client_hash_id using GroupShuffleSplit — no client's pages appear in both train and test, so the split tests generalization to unseen clients, not memorized ones.

In [2]:
# ---- Feature list (leakage-checked) ----
# NOTE: ctr_first_half / ctr_second_half deliberately excluded — tested, caused
# Precision@50 = 100% because is_declining IS first-half-vs-second-half clicks.
# See Section 4 for the documented ablation.
feature_cols = [
    'total_gsc_impressions_mar', 'avg_gsc_position_mar',
    'days_with_impressions', 'position_std_mar',
    'content_type', 'word_count', 'char_count', 'search_volume', 'competition',
    'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count',
    'content_age_days', 'days_since_last_update'
]

X = merged[feature_cols].copy()
y = merged['is_declining'].copy()
groups = merged['client_hash_id']

# ---- Client-based split ----
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Client overlap (should be empty):", set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))

# ---- Missing value handling: _missing flags + train-only median fill ----
missing_cols = ['position_std_mar', 'word_count', 'char_count', 'search_volume',
                 'competition', 'cpc', 'backlinks']

for col in missing_cols:
    X_train[f'{col}_missing'] = X_train[col].isna().astype(int)
    X_test[f'{col}_missing'] = X_test[col].isna().astype(int)

for col in missing_cols:
    median_val = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_val)
    X_test[col] = X_test[col].fillna(median_val)

# ---- Categorical encoding ----
categorical_cols = ['content_type', 'competition_level', 'main_intent']
for col in categorical_cols:
    X_train[col] = X_train[col].fillna('unknown')
    X_test[col] = X_test[col].fillna('unknown')

X_train_encoded = pd.get_dummies(X_train, columns=categorical_cols)
X_test_encoded = pd.get_dummies(X_test, columns=categorical_cols)
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

# ---- ctr_mar / ctr_gap: same non-leaky signal the baseline itself uses ----
bins = [0, 3, 10, 20, float("inf")]
tier_labels = ["1-3", "4-10", "11-20", "21+"]
expected_ctr_map = {"1-3": 0.00406253, "4-10": 0.00323895, "11-20": 0.00305241, "21+": 0.00133142}

for split_X, idx in [(X_train_encoded, X_train.index), (X_test_encoded, X_test.index)]:
    tier = pd.cut(merged.loc[idx, "avg_gsc_position_mar"], bins=bins, labels=tier_labels)
    ctr_mar = merged.loc[idx, "total_gsc_clicks_mar"] / merged.loc[idx, "total_gsc_impressions_mar"]
    split_X["ctr_mar"] = ctr_mar.values
    split_X["ctr_gap"] = (tier.map(expected_ctr_map).astype(float) - ctr_mar).values

print("Final feature count:", X_train_encoded.shape[1])

Train: (94371, 16)  Test: (7038, 16)
Client overlap (should be empty): set()
Final feature count: 34


## 3. Train + compare vs my baseline

Random Forest trained on the leakage-checked feature set, evaluated at Precision@50 on the held-out test split. Baseline recomputed on the exact same test rows (not the full Week 4 dataset) so the comparison is apples-to-apples.

Random Forest, evaluated via 5-fold GroupKFold cross-validation (to control for client-size skew seen in single random splits), achieves an average Precision@50 of 52% versus the baseline's 46% on the same real-volume subset (base rate 26.4%, matching Week 4's signal audit) — a lift of 1.97x vs baseline's 1.74x. The improvement is directional but not uniform: RF outperformed the baseline in 3 of 5 folds, with the baseline winning in the remaining 2, indicating the RF's edge is real but modest and sensitive to client composition.

In [3]:
# ---- Train Random Forest ----
rf = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=20,
    random_state=42, n_jobs=-1
)
rf.fit(X_train_encoded, y_train)

rf_probs = rf.predict_proba(X_test_encoded)[:, 1]
eval_rf = X_test_encoded.copy()
eval_rf["is_declining"] = y_test.values
eval_rf["rf_score"] = rf_probs
eval_rf = eval_rf.sort_values("rf_score", ascending=False)

k = 50
precision_at_50_rf = eval_rf.head(k)["is_declining"].mean()
base_rate_test = eval_rf["is_declining"].mean()

print(f"RF Precision@50: {precision_at_50_rf:.2%}")
print(f"Base rate (test set): {base_rate_test:.2%}")
print(f"RF Lift over random: {precision_at_50_rf / base_rate_test:.2f}x")

# ---- Baseline, recomputed on the SAME test set ----
baseline_test = merged.loc[X_test.index].copy()
baseline_test["position_tier"] = pd.cut(baseline_test["avg_gsc_position_mar"], bins=bins, labels=tier_labels)
baseline_test["expected_ctr"] = baseline_test["position_tier"].map(expected_ctr_map).astype(float)
baseline_test["ctr_mar"] = baseline_test["total_gsc_clicks_mar"] / baseline_test["total_gsc_impressions_mar"]
baseline_test["ctr_gap"] = baseline_test["expected_ctr"] - baseline_test["ctr_mar"]
baseline_test["score"] = baseline_test["ctr_gap"].clip(lower=0) * baseline_test["total_gsc_impressions_mar"]
baseline_test = baseline_test.sort_values("score", ascending=False)

precision_at_50_baseline = baseline_test.head(k)["is_declining"].mean()
print(f"\nBaseline Precision@50 (same test set): {precision_at_50_baseline:.2%}")
print(f"Baseline Lift over random: {precision_at_50_baseline / base_rate_test:.2f}x")

# ---- Comparison table ----
comparison = pd.DataFrame({
    "Model": ["Baseline (rule-based)", "Random Forest"],
    "Precision@50": [precision_at_50_baseline, precision_at_50_rf],
    "Base rate": [base_rate_test, base_rate_test],
    "Lift": [precision_at_50_baseline / base_rate_test, precision_at_50_rf / base_rate_test]
})
print("\n", comparison)

RF Precision@50: 24.00%
Base rate (test set): 22.35%
RF Lift over random: 1.07x

Baseline Precision@50 (same test set): 22.00%
Baseline Lift over random: 0.98x

                    Model  Precision@50  Base rate      Lift
0  Baseline (rule-based)          0.22   0.223501  0.984336
1          Random Forest          0.24   0.223501  1.073821


In [4]:
importances_v2 = pd.Series(rf.feature_importances_, index=X_train_encoded.columns)
print(importances_v2.sort_values(ascending=False).head(10))

auc_v2 = roc_auc_score(y_test, rf_probs)
print(f"\nROC-AUC: {auc_v2:.3f}")

ctr_mar                      0.426837
ctr_gap                      0.226293
total_gsc_impressions_mar    0.096393
days_with_impressions        0.088773
content_age_days             0.044464
position_std_mar             0.035396
avg_gsc_position_mar         0.033324
days_since_last_update       0.010411
word_count                   0.007413
char_count                   0.005848
dtype: float64

ROC-AUC: 0.809


In [5]:
def run_split_and_evaluate(seed):
    # ---- Split ----
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    train_idx, test_idx = next(gss.split(X, y, groups=groups))
    Xtr, Xte = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
    ytr, yte = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

    # ---- Missing value handling (train-only stats, har baar dobara seekha jata hai) ----
    for col in missing_cols:
        Xtr[f'{col}_missing'] = Xtr[col].isna().astype(int)
        Xte[f'{col}_missing'] = Xte[col].isna().astype(int)
    for col in missing_cols:
        med = Xtr[col].median()
        Xtr[col] = Xtr[col].fillna(med)
        Xte[col] = Xte[col].fillna(med)

    # ---- Categorical encoding ----
    for col in categorical_cols:
        Xtr[col] = Xtr[col].fillna('unknown')
        Xte[col] = Xte[col].fillna('unknown')
    Xtr_enc = pd.get_dummies(Xtr, columns=categorical_cols)
    Xte_enc = pd.get_dummies(Xte, columns=categorical_cols)
    Xtr_enc, Xte_enc = Xtr_enc.align(Xte_enc, join='left', axis=1, fill_value=0)

    # ---- ctr_mar / ctr_gap ----
    for split_X, idx in [(Xtr_enc, Xtr.index), (Xte_enc, Xte.index)]:
        tier = pd.cut(merged.loc[idx, "avg_gsc_position_mar"], bins=bins, labels=tier_labels)
        ctr_mar = merged.loc[idx, "total_gsc_clicks_mar"] / merged.loc[idx, "total_gsc_impressions_mar"]
        split_X["ctr_mar"] = ctr_mar.values
        split_X["ctr_gap"] = (tier.map(expected_ctr_map).astype(float) - ctr_mar).values

    # ---- Train RF ----
    rf_s = RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20,
                                   random_state=42, n_jobs=-1)
    rf_s.fit(Xtr_enc, ytr)
    probs = rf_s.predict_proba(Xte_enc)[:, 1]

    eval_s = Xte_enc.copy()
    eval_s["is_declining"] = ytr_check = yte.values
    eval_s["score"] = probs
    eval_s = eval_s.sort_values("score", ascending=False)
    rf_p50 = eval_s.head(50)["is_declining"].mean()
    base_rate_s = eval_s["is_declining"].mean()

    # ---- Baseline, same test rows ----
    bt = merged.loc[Xte.index].copy()
    bt["position_tier"] = pd.cut(bt["avg_gsc_position_mar"], bins=bins, labels=tier_labels)
    bt["expected_ctr"] = bt["position_tier"].map(expected_ctr_map).astype(float)
    bt["ctr_mar"] = bt["total_gsc_clicks_mar"] / bt["total_gsc_impressions_mar"]
    bt["ctr_gap"] = bt["expected_ctr"] - bt["ctr_mar"]
    bt["score"] = bt["ctr_gap"].clip(lower=0) * bt["total_gsc_impressions_mar"]
    bt = bt.sort_values("score", ascending=False)
    base_p50 = bt.head(50)["is_declining"].mean()

    return {"seed": seed, "test_size": len(Xte), "base_rate": base_rate_s,
            "baseline_p50": base_p50, "rf_p50": rf_p50}

# ---- 3 alag seeds pe chalao ----
results = [run_split_and_evaluate(s) for s in [42, 7, 99]]
results_df = pd.DataFrame(results)
print(results_df)

   seed  test_size  base_rate  baseline_p50  rf_p50
0    42       7038   0.223501          0.22    0.24
1     7      30044   0.245973          0.52    0.54
2    99       1496   0.196524          0.32    0.46


In [6]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)
fold_results = []

for fold_num, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=groups)):
    Xtr, Xte = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
    ytr, yte = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

    for col in missing_cols:
        Xtr[f'{col}_missing'] = Xtr[col].isna().astype(int)
        Xte[f'{col}_missing'] = Xte[col].isna().astype(int)
    for col in missing_cols:
        med = Xtr[col].median()
        Xtr[col] = Xtr[col].fillna(med)
        Xte[col] = Xte[col].fillna(med)

    for col in categorical_cols:
        Xtr[col] = Xtr[col].fillna('unknown')
        Xte[col] = Xte[col].fillna('unknown')
    Xtr_enc = pd.get_dummies(Xtr, columns=categorical_cols)
    Xte_enc = pd.get_dummies(Xte, columns=categorical_cols)
    Xtr_enc, Xte_enc = Xtr_enc.align(Xte_enc, join='left', axis=1, fill_value=0)

    for split_X, idx in [(Xtr_enc, Xtr.index), (Xte_enc, Xte.index)]:
        tier = pd.cut(merged.loc[idx, "avg_gsc_position_mar"], bins=bins, labels=tier_labels)
        ctr_mar = merged.loc[idx, "total_gsc_clicks_mar"] / merged.loc[idx, "total_gsc_impressions_mar"]
        split_X["ctr_mar"] = ctr_mar.values
        split_X["ctr_gap"] = (tier.map(expected_ctr_map).astype(float) - ctr_mar).values

    rf_f = RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20,
                                   random_state=42, n_jobs=-1)
    rf_f.fit(Xtr_enc, ytr)
    probs = rf_f.predict_proba(Xte_enc)[:, 1]

    eval_f = Xte_enc.copy()
    eval_f["is_declining"] = yte.values
    eval_f["score"] = probs
    eval_f = eval_f.sort_values("score", ascending=False)
    rf_p50 = eval_f.head(50)["is_declining"].mean()
    base_rate_f = eval_f["is_declining"].mean()

    bt = merged.loc[Xte.index].copy()
    bt["position_tier"] = pd.cut(bt["avg_gsc_position_mar"], bins=bins, labels=tier_labels)
    bt["expected_ctr"] = bt["position_tier"].map(expected_ctr_map).astype(float)
    bt["ctr_mar"] = bt["total_gsc_clicks_mar"] / bt["total_gsc_impressions_mar"]
    bt["ctr_gap"] = bt["expected_ctr"] - bt["ctr_mar"]
    bt["score"] = bt["ctr_gap"].clip(lower=0) * bt["total_gsc_impressions_mar"]
    bt = bt.sort_values("score", ascending=False)
    base_p50 = bt.head(50)["is_declining"].mean()

    fold_results.append({"fold": fold_num, "test_size": len(Xte),
                          "base_rate": base_rate_f, "baseline_p50": base_p50, "rf_p50": rf_p50})

fold_df = pd.DataFrame(fold_results)
print(fold_df)
print("\nAverage baseline P@50:", fold_df["baseline_p50"].mean())
print("Average RF P@50:", fold_df["rf_p50"].mean())

   fold  test_size  base_rate  baseline_p50  rf_p50
0     0      21633   0.290020          0.50    0.56
1     1      19984   0.288080          0.70    0.50
2     2      19931   0.283578          0.50    0.60
3     3      19931   0.261452          0.26    0.64
4     4      19930   0.195785          0.34    0.30

Average baseline P@50: 0.45999999999999996
Average RF P@50: 0.52


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [7]:
# ---- Diagnostics ----
auc = roc_auc_score(y_test, rf_probs)
print(f"ROC-AUC: {auc:.3f}")

importances = pd.Series(rf.feature_importances_, index=X_train_encoded.columns)
print("\nTop 10 feature importances:")
print(importances.sort_values(ascending=False).head(10))

# ---- Leakage ablation test (documented — do not re-run, keep as a record) ----
# ctr_first_half / ctr_second_half were tested as features. Result: Precision@50 = 100%,
# with those two columns alone accounting for ~77% of feature importance.
# Root cause: is_declining is itself (second_half_clicks < first_half_clicks), so
# per-half CTR values (and even their _missing flags) are near-duplicates of the label.
# Both were removed before the results reported above.

ROC-AUC: 0.809

Top 10 feature importances:
ctr_mar                      0.426837
ctr_gap                      0.226293
total_gsc_impressions_mar    0.096393
days_with_impressions        0.088773
content_age_days             0.044464
position_std_mar             0.035396
avg_gsc_position_mar         0.033324
days_since_last_update       0.010411
word_count                   0.007413
char_count                   0.005848
dtype: float64


## Interpretation
Random Forest, using the leakage-free feature set, matches the rule-based baseline's Precision@50 (22% vs 22%, ~0.98x lift over random) on the same client-grouped held-out split. ROC-AUC (0.679) indicates the model captures some genuine signal across the full ranking, but this does not translate into an improvement at the strict top-50 cutoff. Reported as an honest null result: added features and non-linear modeling did not outperform the simpler rule for this ranking task.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.